In [ ]:
import sys
import site

site.addsitedir(site.getusersitepackages())
import scipy
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import sklearn
from scipy.io import loadmat
import sktime

In [ ]:
## Directory of all .MAT files
matdata_dir = "./data/matfiles"

## Disposition
disposition = pd.read_csv("./data/disposition.csv")

## All analyses should only use files marked "Reduced"
reduced_disp = disposition[disposition["Reduced"] == "X"]

## Add .mat to DAQs so they can be read from the mat data directory, reset index to facilitate looping
reduced_disp = reduced_disp.assign(
    MatName=reduced_disp["DaqName"].str[:-4] + ".mat"
).reset_index(0, drop=True)

In [ ]:
## Empty final df
final_df = pd.DataFrame(
    columns=[
        "Subject",
        "Drive",
        "LPupil_Diameter",
        "RPupil_Diameter",
        "Lateral_Deviation",
        "Sample",
        "Blink_Duration",
        "Left_Gaze_Dir_X",
        "Left_Gaze_Dir_Y",
        "Left_Gaze_Dir_Z",
        "Right_Gaze_Dir_X",
        "Right_Gaze_Dir_Y",
        "Right_Gaze_Dir_Z",
        "Gaze_Pitch",
        "Gaze_Yaw",
        "Left_Eyelid_Closed",
        "Right_Eyelid_Closed",
        "Vehicle_Speed",
        "Brake_Pedal_Force",
        "Steering_Wheel_Angle",
        "Steering_Wheel_Angle_Rate",
        "Accelerator_Pedal_Position",
        "Head_Pos_X",
        "Head_Pos_Y",
        "Head_Pos_Z",
        "Blink_Counter",
        "Blink_Frequency",
        "Right_Button_Press",
        "Left_Button_Press",
        "DaqName",
    ]
)

# Read in each mat file and
for i in range(0, reduced_disp.shape[0] - 1):

    ## Read current mat file
    fname = matdata_dir + "/" + reduced_disp["MatName"][i]
    try:
        data = loadmat(fname)
        print(f"Successfully read {fname} ({i+1}/{reduced_disp.shape[0] - 1})")
    except FileNotFoundError:
        print(f"Skipping {fname}: File not found ({i+1}/{reduced_disp.shape[0] - 1})")

    ## Vars to keep (just lane dev and pupil diam for now)

    variable_names = {
        "LPupil_Diameter": "Output_FovioDMEResults_dme_core_pupil_left_pupil_diameter_mm",
        "RPupil_Diameter": "Output_FovioDMEResults_dme_core_pupil_right_pupil_diameter_mm",
        "Blink_Counter": "Output_FovioDMEResults_dme_core_eyelid_blink_counter",
        "Blink_Frequency": "Output_FovioDMEResults_dme_core_eyelid_blink_frequency",
        "Head_Pos_X": "Output_FovioDMEResults_dme_core_head_head_pose_translation_x",
        "Head_Pos_Y": "Output_FovioDMEResults_dme_core_head_head_pose_translation_y",
        "Head_Pos_Z": "Output_FovioDMEResults_dme_core_head_head_pose_translation_z",
        "Blink_Duration": "Output_FovioDMEResults_dme_core_eyelid_average_blink_duration_u",
        "Left_Gaze_Dir_X": "Output_FovioDMEResults_dme_core_gaze_left_eye_gaze_direction_x",
        "Left_Gaze_Dir_Y": "Output_FovioDMEResults_dme_core_gaze_left_eye_gaze_direction_y",
        "Left_Gaze_Dir_Z": "Output_FovioDMEResults_dme_core_gaze_left_eye_gaze_direction_z",
        "Right_Gaze_Dir_X": "Output_FovioDMEResults_dme_core_gaze_right_eye_gaze_direction_x",
        "Right_Gaze_Dir_Y": "Output_FovioDMEResults_dme_core_gaze_right_eye_gaze_direction_y",
        "Right_Gaze_Dir_Z": "Output_FovioDMEResults_dme_core_gaze_right_eye_gaze_direction_z",
        "Gaze_Pitch": "Output_FovioDMEResults_dgaze_unified_gaze_direction_deg_pitch_d",
        "Gaze_Yaw": "Output_FovioDMEResults_dgaze_unified_gaze_direction_deg_yaw_deg",
        "Right_Eyelid_Closed": "Output_FovioDMEResults_dme_core_eyelid_right_eyelid_closed",
        "Left_Eyelid_Closed": "Output_FovioDMEResults_dme_core_eyelid_left_eyelid_closed",
        "Vehicle_Speed": "VDS_Veh_Speed",
        "Brake_Pedal_Force": "CFS_Brake_Pedal_Force",
        "Steering_Wheel_Angle": "CFS_Steering_Wheel_Angle",
        "Steering_Wheel_Angle_Rate": "CFS_Steering_Wheel_Angle_Rate",
        "Accelerator_Pedal_Position": "CFS_Accelerator_Pedal_Position",
        "Left_Button_Press": "HI_LeftExtraButtons_QuarterCab",
        "Right_Button_Press": "HI_RightExtraButtons_QuarterCab",
    }

    variable_series = {}

    ## Full lat pos series (shaped differently so taken differently)
    variable_series["Lateral_Deviation"] = data["elemDataI"]["SCC_Lane_Deviation"][0][0][:, 1].flatten()
    nstep = variable_series["Lateral_Deviation"].shape[0]

    for key, val in variable_names.items():
        try:
            variable_series[key] = data["elemDataI"][val][0][0].flatten()
        except ValueError:
            variable_series[key] = np.repeat(99, nstep)

    ## Assemble DF w/ Subject and Drive meta data
    temp_df = pd.DataFrame(variable_series)
    temp_df.insert(0, "Subject", np.repeat(reduced_disp["Subject"][i], nstep))
    temp_df.insert(1, "Drive", np.repeat(reduced_disp["DriveN"][i], nstep))
    temp_df["DaqName"] = np.repeat(reduced_disp["MatName"][i][:-4], nstep)

    ## Attach "Sample" for each 60 sec sample
    temp_df["Sample"] = 1 + temp_df.index // 3600

    ## Discard any samples that aren't length 3600
    # NOTE might not want to discard these if we aren't using samples
    temp_df = temp_df[temp_df.groupby("Sample")["Sample"].transform("size") == 3600]

    ## Append the temp df to the current final df
    final_df = pd.concat([final_df, temp_df], ignore_index=True)

final_df["Sample_ID"] = (
    "ID_"
    + final_df["Subject"].astype(str)
    + "_"
    + final_df["Drive"].astype(str)
    + "_"
    + final_df["Sample"].astype(str)
)

print("Writing csv...")
final_df.to_csv("./data/non_aug_data.csv")